# 02. Biomedical Named Entity Recognition (NER) & Code Mapping
### Clinical Note-to-Billing Code Consistency Checker (FWA Triage)

Here we extract diagnoses (DISEASE), medications (CHEMICAL), procedures (PROCEDURE), and anatomical sites (ANATOMY) from clinical notes and map them to standardized ICD-10 / HCPCS code categories.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from clinical_billing_audit.nlp.ner_extractor import NERExtractor
from clinical_billing_audit.nlp.entity_normalizer import normalize_entity
from clinical_billing_audit.nlp.code_mapper import entity_to_code_category, predict_code_categories_from_entities

## 1. Extract Entities from Sample Note

In [ ]:
extractor = NERExtractor()

sample_note = (
    "HISTORY: 64-year-old male with severe essential hypertension and chronic congestive heart failure. "
    "Presents with worsening dyspnea. Underwent diagnostic colonoscopy showing diverticulosis. "
    "Current medications include furosemide 40 mg daily, lisinopril 20 mg, and aspirin."
)

result = extractor.extract(sample_note, note_id="DEMO_01")
print(f"Extracted {len(result.entities)} entities:")
print(f"- Diagnoses:   {result.diagnoses}")
print(f"- Medications: {result.medications}")
print(f"- Procedures:  {result.procedures}")
print(f"- Anatomy:     {result.anatomy}")

## 2. Entity Normalization & Code Category Mapping

In [ ]:
normalized_diags = [normalize_entity(d, "DISEASE") for d in result.diagnoses]
normalized_procs = [normalize_entity(p, "PROCEDURE") for p in result.procedures]

print("Normalized Diagnoses:", normalized_diags)
print("Normalized Procedures:", normalized_procs)

category_votes = predict_code_categories_from_entities(normalized_diags, normalized_procs)
print("\nPredicted Code Categories (from entities):")
for cat, score in category_votes.items():
    print(f"  {cat:<25}: {score:.1%}")